# 创建一个 Llava 模型

本 notebook 将基于 clip-vit-large-patch-14-336 和 Qwen1.5-4b-Chat 模型，搭建 Llava-1.5 架构。

In [ ]:
# pip install transformers==4.57.6 huggingface_hub modelscope

## 下载模型

和 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中一样，我们先下载好模型。如果镜像站下载太慢，建议使用魔塔社区下载。

```bash

pip install -U huggingface_hub

export HF_ENDPOINT=https://hf-mirror.com

huggingface-cli download --resume-download openai/clip-vit-large-patch14-336 --local-dir openai/clip-vit-large-patch14-336 --local-dir-use-symlinks False


huggingface-cli download --resume-download Qwen/Qwen1.5-4B-Chat --local-dir Qwen1.5-4B-Chat --local-dir-use-symlinks False
```

### 使用魔塔社区下载

In [ ]:
use_modelscope=True

In [ ]:
if use_modelscope:
    # 安装或更新 modelscope 库（如果环境中没有的话）
    # pip install -U modelscope
    
    # 从魔搭下载模型
    from modelscope import snapshot_download
    
    model_dir = snapshot_download(
        'AI-ModelScope/clip-vit-large-patch14-336',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
    
    print(f"vision 模型已下载到: {model_dir}")
    
    model_dir = snapshot_download(
        'Qwen/Qwen1.5-4B-Chat',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
        
    print(f"LLM 模型已下载到: {model_dir}")

## 修改 Qwen tokenizer

原始的 Qwen Tokenizer 中是不包含 Llava 中引入的 `<image>` 词元的，如果直接使用的话会被拆成三个 token：`<`、`image`、`>`。为了让 Qwen 能够识别我们引入的多模态 `<image>` 词元，需要我们手动添加到下载的文件中的`tokenizer_config.json`中：

1. 在`added_tokens_decoder`里面，加上：
   ```json
    "151646": {
        "content": "<image>",
        "lstrip": false,
        "normalized": false,
        "rstrip": false,
        "single_word": false,
        "special": true
        }
    ```
2. 在`additional_special_tokens` 里面加上 `"<image>"`
   
添加之后简单做一下测试，注意替换模型路径为上面的下载代码打印出来的 LLM 路径。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 请替换路径
modify_qwen_tokenizer_dir = "/home/yuanz/documents/models/Qwen1.5-4B-Chat"
modify_qwen_tokenizer = AutoTokenizer.from_pretrained(modify_qwen_tokenizer_dir)

modify_qwen_tokenizer.encode("<image>")

### 加了这个新 token，需要修改模型的 embedding 模块么？

不需要。这是使用 Qwen 模型的好处。Qwen 在训练时给 embedding 层预留了较大的空间，我们添加一些特殊词元不会导致报错。

In [ ]:
qwen_model = AutoModelForCausalLM.from_pretrained(modify_qwen_tokenizer_dir, device_map='cuda:0', 
                                                  dtype=torch.bfloat16)

# 打印添加 <image> 后 Qwen tokenizer 的长度
print(len(modify_qwen_tokenizer))

# 打印 Qwen Embedding 维度
print((qwen_model.model.embed_tokens,))

可以看到 embed_tokens 的长度比 tokenizer 的长度要大。

## Llava 模型初始化

根据上面的下载代码打印的路径，配置 Vision Model 和 Language Model 的路径。

In [ ]:
# 请修改成你的路径
clip_model_name_or_path = (
    "/home/yuanz/documents/models/openai/clip-vit-large-patch14-336"
)
qwen_model_name_or_path = "/home/yuanz/documents/models/Qwen1.5-4B-Chat"

使用 transformers 接口，将两个模型读入内存。（这里直接读入显存，如果 OOM 把 device 改回 cpu）

In [ ]:
from transformers import AutoModel, AutoModelForCausalLM

clip_model = AutoModel.from_pretrained(
    clip_model_name_or_path,
    device_map="cuda:0",
)
llm_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name_or_path,
    device_map="cuda:0",
    dtype=torch.bfloat16,
)

使用内存中的模型，初始化 llava 模型的配置，并创建一个 llava 模型对象。

In [ ]:
from transformers import LlavaForConditionalGeneration, LlavaConfig

# 初始化 CLIP-vision config
vision_config = clip_model.vision_model.config

# 初始化 LLM config
text_config = llm_model.config

# 用上面的 config 初始化 llava-1.5-7b style configuration
configuration = LlavaConfig(vision_config, text_config)

# 使用 llava config 创建 llava 模型
model = LlavaForConditionalGeneration(configuration)

作为检验，我们看看 Llava 中的 embedding 层和 CLIP 中的 embedding 层，检查形状信息是否成功导入 Llava 配置。

In [ ]:
model.vision_tower.vision_model.embeddings

In [ ]:
clip_model.vision_model.embeddings

将权重复制到 llava 模型中。

In [ ]:
# 更换视觉模型
model.vision_tower.vision_model = clip_model.vision_model

# 更换语言模型
model.set_decoder(llm_model.model)
model.lm_head = llm_model.lm_head

作为检验，我们看看 Qwen 的 embedding 权重和 Llama 的 embedding 权重，检查是否赋值成功。

In [ ]:
llm_model.model.embed_tokens.weight.data[:, :2]

In [ ]:
model.language_model.embed_tokens.weight.data[:, :2]

在 Llava 模型中还有一些属性需要配置，因为使用构造函数初始化 LlavaConfig 时并不会自动设置。这里主要关注 `pad_token_id` 和 `image_token_index`。这一步提前到直接对 configuration 进行修改也可以：

```python
configuration.pad_token_id = text_config.pad_token_id  # 151643
```

In [ ]:
model.config.pad_token_id = llm_model.config.pad_token_id  # 151643
model.config.eos_token_id = llm_model.config.eos_token_id
# image_token_index 在下一步 tokenizer 初始化后设置，因为需要 tokenizer.encode("<image>")[0]

## Llava Tokenizer 初始化

我们用 Qwen 替换了原始模型的 Vicuna，所以需要同步修改 tokenizer。这一步的目标是：把已经改好的 Qwen tokenizer 和 CLIP 的 image processor 组装成一个真正的 LlavaProcessor，并写入 Llava 推理必需的元数据。

首先读取我们添加 `<image>` 后的 Qwen Tokenizer。加载后验证是否能够读取特殊词元。

In [ ]:
from transformers import AutoTokenizer

llm_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name_or_path)
image_token_id = llm_tokenizer.encode("<image>")[0] # 取 [0] 因为 encoder 默认返回 list
# 应该输出 151646，而不是被拆成多个 token

将特殊 token id 加入 Llava 模型的配置文件中。

In [ ]:
model.config.image_token_index = image_token_id

对于 CLIP 的 processor，与参考仓库不同，`LlavaProcessor` 不是简单地把 CLIP 的 `preprocessor_config.json` 复制过来就能用的。它需要知道**每张图片会产生多少视觉 token**，才能正确插入 `<image>` 占位符。这三个属性在新版 `transformers` 中是**显式读取的**：

| 属性 | 值 | 原因 |
|---|---|---|
| `patch_size` | `14` | CLIP-ViT-L/14 的 patch 大小 |
| `vision_feature_select_strategy` | `"default"` | 取 CLIP 倒数第二层特征，并去掉 CLS token |
| `num_additional_image_tokens` | `1` | CLIP 会额外产生一个 CLS token，需要计入总 token 数 |

In [ ]:
from transformers import LlavaProcessor, CLIPImageProcessor

image_processor = CLIPImageProcessor.from_pretrained(clip_model_name_or_path)

llava_processor = LlavaProcessor(
    image_processor=image_processor,
    tokenizer=llm_tokenizer,
)

# 同样的，构造函数不会默认设置这些数据。显式写入 Llava 推理所需的元数据
llava_processor.patch_size = model.config.vision_config.patch_size  # 14
llava_processor.vision_feature_select_strategy = "default"
llava_processor.num_additional_image_tokens = 1     # CLIP 的 CLS token

保存模型 + Tokenizer + Processor 到同一个目录。

In [ ]:
save_dir = "show_model/model001"

model.save_pretrained(save_dir)
# 实际上 llava_processor 会自己保存自己的 llm_tokenizer。
# 这里是为了可读性和冗余设计。
llm_tokenizer.save_pretrained(save_dir)
llava_processor.save_pretrained(save_dir)

保存后目录里应该同时包含：

- `config.json`（LlavaConfig，含 `image_token_index`）
- `model.safetensors` 或分片权重
- `tokenizer_config.json`（含 `<image>` 注册）
- `preprocessor_config.json`（含 `processor_class: "LlavaProcessor"`、`patch_size`、`vision_feature_select_strategy`、`num_additional_image_tokens`）

## 测试效果

在 notebook 中**重启内核后直接运行**下面的代码（跳过上方的代码块），测试之前保存的模型能不能直接使用。

In [ ]:
from transformers import LlavaProcessor, LlavaForConditionalGeneration
import torch

model_name_or_path = "show_model/model001"

llava_processor = LlavaProcessor.from_pretrained(model_name_or_path)
model = LlavaForConditionalGeneration.from_pretrained(
    model_name_or_path, device_map="cuda:0", dtype=torch.bfloat16
)

进行一次前向推理。注意此时连接视觉模型和语言模型的 MLP 权重是默认初始化的权重，我们将在后续的训练中让网络学习这部分权重。这里我们采用有 system 字段的模格式，因此需要手动在 messages 中指定（原因在 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中）。

In [ ]:
from PIL import Image

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}] # 结构化格式
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What is in this image?"}
        ]
    }
]
image_path = "000000039769.jpg"
image = Image.open(image_path)

prompt = llava_processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = llava_processor(text=prompt, images=image, return_tensors="pt").to(
    model.device, dtype=torch.bfloat16
)

generate_ids = model.generate(**inputs, max_new_tokens=20)
# 这里保留特殊 token，用于观察输出
gen_text = llava_processor.batch_decode(
    generate_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
)[0]    # 取 0 因为 batch_decode 支持批量解码，第一个维度是 batch 维

print(gen_text)

可以打印输入 prompt 看看 Qwen 的输入格式。

In [ ]:
print(prompt)